# TIỀN XỬ LÝ DỮ LIỆU: AQI VÀ BỆNH HÔ HẤP (COPD, CURRENT ASTHMA) THEO COUNTY, 2019–2022

**Ghi chú phiên bản (v3), các thay đổi chính:**
1. **Không xóa outlier.** IQR ở mục 5.1 chỉ dùng để mô tả; các quan sát vượt ngưỡng được **đánh dấu** bằng cột `is_extreme_*` (mục 9.3), tính trên bộ dữ liệu phân tích cuối cùng (sau khi lọc độ phủ AQI và City). Cờ AQI được tách thành phía **cao** và **thấp**.
2. Kiểm tra `Disease_AdjPrev` / `Smoking_AdjPrev` có đổi theo năm không và độ lớn của thay đổi (mục 4.4–4.6).
3. Ghi rõ nguồn dữ liệu và cách Join AQI với PLACES (notebook `Merge_AQI_PLACES.ipynb`).
4. Cố định một toạ độ cho mỗi County (mục 9.2); đường dẫn và tham số gom vào ô cấu hình; có bảng theo dõi số dòng (mục 11.3) và cell kiểm tra tổng hợp (mục 11.2).
5. Bổ sung từ điển dữ liệu cho file bàn giao (mục 13).

IMPORT CÁC THƯ VIỆN VÀO

In [39]:
import os
import pandas as pd
import numpy as np

CẤU HÌNH ĐƯỜNG DẪN VÀ THAM SỐ

In [40]:
# ---- Cấu hình: chỉ sửa ở đây ----
DATA_DIR         = r"D:\TT Dữ liệu trực quan\ProjectCuoiKy\Dataset\AQI — EPA AirData"
MERGED_FILE      = os.path.join(DATA_DIR, "AQI_Respiratory_Disease_2019_2022_Merged_Clean.csv")
AQS_SITES_FILE   = "aqs_sites.csv"                       # file AQS Sites của EPA (đặt cùng thư mục notebook hoặc ghi đường dẫn đầy đủ)
OUTPUT_FILE      = "AQI_Respiratory_Disease_Final.csv"   # file bàn giao
MIN_COVERAGE_PCT = 50                                    # bỏ County-Year có độ phủ AQI thấp hơn ngưỡng này (%)

1. ĐỌC DỮ LIỆU ĐẦU VÀO

**Nguồn dữ liệu đầu vào:** file `AQI_Respiratory_Disease_2019_2022_Merged_Clean.csv` được tạo từ notebook `Merge_AQI_PLACES.ipynb`, ghép:
- **AQI gốc** (EPA AirData, `annual_aqi_by_county_2019–2022`), cấp County-Year;
- **PLACES gốc** (CDC), lấy 3 chỉ số age-adjusted prevalence: hút thuốc (`CSMOKING`), COPD, hen hiện tại (`CASTHMA`).

Khóa ghép: `CountyFIPS` + `Year`. Mỗi County-Year xuất hiện 2 dòng (COPD và Current Asthma). Chi tiết số dòng trước/sau Join xem trong notebook Merge.

In [41]:
df = pd.read_csv(MERGED_FILE)

# Bảng theo dõi số dòng qua từng bước (in ở mục 11.3)
pipeline_log = [("Đọc file Merged_Clean", len(df), df["CountyFIPS"].nunique())]

print("Shape:", df.shape)
df.head()

Shape: (7566, 24)


,CountyFIPS,County,StateAbbr,State,Year,Population,Geolocation,Days_with_AQI,Good_Days,Moderate_Days,...,Max_AQI,AQI_90th_Percentile,Median_AQI,Days_NO2,Days_Ozone,Days_PM2_5,Days_PM10,Smoking_AdjPrev,Disease,Disease_AdjPrev
0,1003,Baldwin,AL,Alabama,2019,"223,234",POINT (-87.72275422 30.72811673),271,221,50,...,80,55,40,0,198,73,0,19.9,COPD,6.9
1,1003,Baldwin,AL,Alabama,2019,"223,234",POINT (-87.72275422 30.72811673),271,221,50,...,80,55,40,0,198,73,0,19.9,Current Asthma,8.9
2,1027,Clay,AL,Alabama,2019,"13,235",POINT (-85.86076142 33.26916978),107,75,32,...,71,56,41,0,0,107,0,25.6,COPD,10.1
3,1027,Clay,AL,Alabama,2019,"13,235",POINT (-85.86076142 33.26916978),107,75,32,...,71,56,41,0,0,107,0,25.6,Current Asthma,10.6
4,1033,Colbert,AL,Alabama,2019,"55,241",POINT (-87.80480288 34.70084917),263,238,25,...,65,50,38,0,219,44,0,21.5,COPD,8.2


2. KIỂM TRA CẤU TRÚC DỮ LIỆU

In [42]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Columns:
['CountyFIPS', 'County', 'StateAbbr', 'State', 'Year', 'Population', 'Geolocation', 'Days_with_AQI', 'Good_Days', 'Moderate_Days', 'USG_Days', 'Unhealthy_Days', 'Very_Unhealthy_Days', 'Hazardous_Days', 'Max_AQI', 'AQI_90th_Percentile', 'Median_AQI', 'Days_NO2', 'Days_Ozone', 'Days_PM2_5', 'Days_PM10', 'Smoking_AdjPrev', 'Disease', 'Disease_AdjPrev']

Data types:
CountyFIPS               int64
County                     str
StateAbbr                  str
State                      str
Year                     int64
Population                 str
Geolocation                str
Days_with_AQI            int64
Good_Days                int64
Moderate_Days            int64
USG_Days                 int64
Unhealthy_Days           int64
Very_Unhealthy_Days      int64
Hazardous_Days           int64
Max_AQI                  int64
AQI_90th_Percentile      int64
Median_AQI               int64
Days_NO2                 int64
Days_Ozone               int64
Days_PM2_5               int64
Days_P

3. KIỂM TRA MISSING VALUE VÀ GIÁ TRỊ TRÙNG LẶP

In [43]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print(
    "Duplicate County-Year-Disease:",
    df.duplicated(subset=["CountyFIPS", "Year", "Disease"]).sum()
)

Missing values:
CountyFIPS             0
County                 0
StateAbbr              0
State                  0
Year                   0
Population             0
Geolocation            0
Days_with_AQI          0
Good_Days              0
Moderate_Days          0
USG_Days               0
Unhealthy_Days         0
Very_Unhealthy_Days    0
Hazardous_Days         0
Max_AQI                0
AQI_90th_Percentile    0
Median_AQI             0
Days_NO2               0
Days_Ozone             0
Days_PM2_5             0
Days_PM10              0
Smoking_AdjPrev        0
Disease                0
Disease_AdjPrev        0
dtype: int64

Duplicate rows: 0
Duplicate County-Year-Disease: 0


4. CHUẨN HÓA DỮ LIỆU

In [44]:
#4.1 Chuẩn hóa biến population
df["Population"] = (
    df["Population"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype("int64")
)

print("Population type:", df["Population"].dtype)
print(df["Population"].head())

Population type: int64
0    223234
1    223234
2     13235
3     13235
4     55241
Name: Population, dtype: int64


In [45]:
# 4.2 Kiểm tra giá trị không hợp lệ

day_cols = [
    "Days_with_AQI", "Good_Days", "Moderate_Days",
    "USG_Days", "Unhealthy_Days",
    "Very_Unhealthy_Days", "Hazardous_Days",
    "Days_NO2", "Days_Ozone", "Days_PM2_5", "Days_PM10"
]

print("Population <= 0:", (df["Population"] <= 0).sum())
print("Số ngày ngoài khoảng 0-366:", ((df[day_cols] < 0) | (df[day_cols] > 366)).sum().sum())
print("Smoking ngoài 0-100:", ((df["Smoking_AdjPrev"] < 0) | (df["Smoking_AdjPrev"] > 100)).sum())
print("Disease ngoài 0-100:", ((df["Disease_AdjPrev"] < 0) | (df["Disease_AdjPrev"] > 100)).sum())

Population <= 0: 0
Số ngày ngoài khoảng 0-366: 0
Smoking ngoài 0-100: 0
Disease ngoài 0-100: 0


In [46]:
# 4.3 Kiểm tra tính nhất quán số ngày AQI

aqi_day_sum = (
    df["Good_Days"]
    + df["Moderate_Days"]
    + df["USG_Days"]
    + df["Unhealthy_Days"]
    + df["Very_Unhealthy_Days"]
    + df["Hazardous_Days"]
)

print("Không khớp Days_with_AQI:", (aqi_day_sum != df["Days_with_AQI"]).sum())

Không khớp Days_with_AQI: 0


**Kiểm tra giới hạn dữ liệu bệnh (PLACES):** dữ liệu PLACES có thể là snapshot của một năm được lặp lại cho nhiều năm AQI. Nếu vậy, `Disease_AdjPrev` không biến thiên theo năm và không được diễn giải là "bệnh thay đổi theo AQI qua các năm". Mục 4.4 kiểm tra điều này cho cả `Disease_AdjPrev` và `Smoking_AdjPrev`.

In [47]:
# 4.4 Disease_AdjPrev / Smoking_AdjPrev có thay đổi theo năm trong cùng County không?

def check_within_county_variation(value_col, group_cols):
    chk = pd.DataFrame({
        "n_years":         df.groupby(group_cols)["Year"].nunique(),
        "n_unique_values": df.groupby(group_cols)[value_col].nunique(),
    })
    multi = chk[chk["n_years"] >= 2]          # chỉ xét County có >= 2 năm
    pct_changed = (multi["n_unique_values"] > 1).mean() * 100
    print(f"--- {value_col} ---")
    print("Số nhóm có >= 2 năm:", len(multi))
    print("Số giá trị khác nhau theo nhóm:")
    print(multi["n_unique_values"].value_counts().sort_index())
    print("Tỷ lệ nhóm có giá trị ĐỔI theo năm:", round(pct_changed, 2), "%")
    return pct_changed

pct_disease = check_within_county_variation("Disease_AdjPrev", ["CountyFIPS", "Disease"])
print()
pct_smoking = check_within_county_variation("Smoking_AdjPrev", ["CountyFIPS"])

print()
if pct_disease < 5:
    print("KẾT LUẬN: Disease_AdjPrev gần như KHÔNG đổi theo năm (snapshot lặp lại).")
    print("-> Chỉ AQI biến thiên theo năm. Không kết luận 'bệnh thay đổi theo AQI qua các năm';")
    print("   phân tích theo lát cắt ngang giữa các County.")
else:
    print("KẾT LUẬN: Disease_AdjPrev có biến thiên theo năm trong cùng County.")

--- Disease_AdjPrev ---
Số nhóm có >= 2 năm: 1964
Số giá trị khác nhau theo nhóm:
n_unique_values
1       2
2     144
3     731
4    1087
Name: count, dtype: int64
Tỷ lệ nhóm có giá trị ĐỔI theo năm: 99.9 %

--- Smoking_AdjPrev ---
Số nhóm có >= 2 năm: 982
Số giá trị khác nhau theo nhóm:
n_unique_values
1      2
2     17
3    211
4    752
Name: count, dtype: int64
Tỷ lệ nhóm có giá trị ĐỔI theo năm: 99.8 %

KẾT LUẬN: Disease_AdjPrev có biến thiên theo năm trong cùng County.


**Kiểm tra cho thấy Disease_AdjPrev và Smoking_AdjPrev thay đổi theo năm** trong cùng County ở hơn 99% các nhóm, nên dữ liệu PLACES không phải snapshot lặp lại và dữ liệu bệnh có chiều thời gian. Tuy nhiên, các giá trị PLACES là ước lượng vùng nhỏ theo mô hình dựa trên BRFSS, nên sự khác biệt giữa các năm có thể phản ánh một phần khác biệt giữa các bản phát hành; các kết luận theo thời gian được diễn giải thận trọng.

In [48]:
# 4.5 Độ lớn thay đổi theo năm của Disease_AdjPrev (đơn vị: điểm phần trăm)
g = df.sort_values("Year").groupby(["CountyFIPS", "Disease"])["Disease_AdjPrev"]

within_std = g.std().dropna()
print("Độ lệch chuẩn trong cùng County (theo bệnh):")
print(within_std.groupby(level="Disease").describe().round(3))

# So sánh với độ chênh giữa các County (cùng một năm)
between_std = df.groupby(["Year", "Disease"])["Disease_AdjPrev"].std().groupby("Disease").mean()
print("\nĐộ lệch chuẩn giữa các County (trung bình các năm):")
print(between_std.round(3))

Độ lệch chuẩn trong cùng County (theo bệnh):
                count   mean    std    min    25%    50%    75%    max
Disease                                                               
COPD            982.0  0.307  0.170  0.000  0.183  0.275  0.395  1.380
Current Asthma  982.0  0.542  0.194  0.058  0.403  0.545  0.680  1.253

Độ lệch chuẩn giữa các County (trung bình các năm):
Disease
COPD              1.454
Current Asthma    0.906
Name: Disease_AdjPrev, dtype: float64


**Kết quả 4.5.** Dữ liệu bệnh chủ yếu khác nhau **giữa các County**, ít khác nhau theo thời gian trong cùng County.

- **COPD:** biến thiên theo năm rất nhỏ. Một County thường chỉ lệch khoảng 0.3 điểm phần trăm quanh mức trung bình của chính nó, trong khi độ lệch chuẩn giữa các County là khoảng 1.45. Ước lượng thô (từ SD²), phần biến thiên theo thời gian chỉ chiếm khoảng 4% tổng phương sai.
- **Hen (Current Asthma):** biến thiên theo năm lớn hơn, bằng khoảng 60% biến thiên giữa các County (0.54 so với 0.91), chiếm khoảng 26% tổng phương sai (ước lượng thô).
- **Ý nghĩa:** tín hiệu chính của bài là so sánh giữa các County. Kết luận kiểu "năm nào AQI tăng thì bệnh tăng theo" sẽ yếu, nhất là với COPD.

*Các số ở 4.5–4.6 tính trên dữ liệu đầu vào, trước khi lọc độ phủ AQI và City.*

In [49]:
# 4.6 Thay đổi theo năm: chung cho mọi County (hiệu ứng năm/bản phát hành) hay riêng từng County?

d = df.copy()
d["Disease_dm"] = d["Disease_AdjPrev"] - d.groupby(["Year", "Disease"])["Disease_AdjPrev"].transform("mean")

def within_between(d, col):
    w = d.groupby(["CountyFIPS", "Disease"])[col].std().dropna().groupby(level="Disease").mean()
    b = d.groupby(["Year", "Disease"])[col].std().groupby("Disease").mean()
    return pd.DataFrame({"within_SD": w, "between_SD": b, "within/between": w / b})

print("Disease_AdjPrev (gốc):")
print(within_between(d, "Disease_AdjPrev").round(3))

print("\nDisease_AdjPrev (đã trừ trung bình từng năm):")
print(within_between(d, "Disease_dm").round(3))

aqi_cy = d.drop_duplicates(["CountyFIPS", "Year"]).assign(Disease="AQI")
print("\nMedian_AQI (để so sánh):")
print(within_between(aqi_cy, "Median_AQI").round(3))

Disease_AdjPrev (gốc):
                within_SD  between_SD  within/between
Disease                                              
COPD                0.307       1.454           0.211
Current Asthma      0.542       0.906           0.598

Disease_AdjPrev (đã trừ trung bình từng năm):
                within_SD  between_SD  within/between
Disease                                              
COPD                0.288       1.454           0.198
Current Asthma      0.322       0.906           0.356

Median_AQI (để so sánh):
         within_SD  between_SD  within/between
Disease                                       
AQI          2.505      10.305           0.243


**Kết quả 4.6.** Sau khi trừ trung bình của từng năm:

- **COPD:** độ lệch chuẩn trong County gần như không đổi (0.307 → 0.288), nên thay đổi theo năm là riêng từng County và nhỏ.
- **Hen:** giảm mạnh (0.542 → 0.322). Ước lượng thô, khoảng hai phần ba (1 − 0.322²/0.542² ≈ 65%) phương sai theo năm là **dịch chuyển chung** của mọi County trong cùng một năm. Nguyên nhân có thể là khác biệt giữa các bản phát hành/phương pháp ước lượng PLACES hoặc một xu hướng chung; dữ liệu này chưa cho phép phân biệt hai khả năng đó.
- **AQI:** tỷ lệ độ lệch chuẩn trong/giữa County là 0.24, cùng cỡ với COPD.

**Kết luận:** cả AQI và bệnh chủ yếu khác nhau giữa các County. Phân tích chính được thực hiện theo lát cắt ngang giữa các County, có kiểm soát hiệu ứng năm (ví dụ thêm `C(Year)` khi hồi quy); phân tích theo thời gian chỉ mang tính bổ sung.

5. MÔ TẢ OUTLIER BẰNG IQR (CHỈ MÔ TẢ, KHÔNG LỌC DÒNG)

In [50]:
# 5.1 Mô tả outlier bằng IQR trên dữ liệu đầu vào (chỉ để mô tả, không lọc dòng)

def iqr_bounds(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

def iqr_summary(data, cols):
    rows = []
    for col in cols:
        q1, q3 = data[col].quantile(0.25), data[col].quantile(0.75)
        lo, hi = iqr_bounds(data[col])
        n_out = int(((data[col] < lo) | (data[col] > hi)).sum())
        rows.append({"Biến": col, "Q1": q1, "Q3": q3, "IQR": q3 - q1,
                     "Lower": lo, "Upper": hi,
                     "Số outlier": n_out, "% outlier": n_out / len(data) * 100})
    return pd.DataFrame(rows).set_index("Biến").round(2)

outlier_cols = [
    "Population", "Days_with_AQI", "Max_AQI", "AQI_90th_Percentile", "Median_AQI",
    "Days_NO2", "Days_Ozone", "Days_PM2_5", "Days_PM10",
    "Smoking_AdjPrev", "Disease_AdjPrev"
]

iqr_summary(df, outlier_cols)

,Q1,Q3,IQR,Lower,Upper,Số outlier,% outlier
Biến,,,,,,,
Population,33608.5,265860.0,232251.5,-314768.75,614237.25,816,10.79
Days_with_AQI,336.0,365.0,29.0,292.50,408.50,1618,21.39
Max_AQI,84.0,133.0,49.0,10.50,206.50,436,5.76
AQI_90th_Percentile,50.0,64.0,14.0,29.00,85.00,460,6.08
Median_AQI,35.0,45.0,10.0,20.00,60.00,526,6.95
Days_NO2,0.0,0.0,0.0,0.00,0.00,1202,15.89
Days_Ozone,60.0,244.0,184.0,-216.00,520.00,0,0.00
Days_PM2_5,8.0,251.0,243.0,-356.50,615.50,0,0.00
Days_PM10,0.0,0.0,0.0,0.00,0.00,1590,21.02


**Lưu ý khi đọc kết quả IQR:** đây chỉ là bảng mô tả trên dữ liệu đầu vào, không có dòng nào bị loại.
- `Days_NO2` và `Days_PM10` có Q1 = Q3 = 0 (IQR = 0), nên mọi giá trị khác 0 đều bị gọi là outlier; với hai biến thưa này, tiêu chí IQR không phù hợp và số "outlier" không mang ý nghĩa thống kê.
- `Max_AQI` lệch phải rất mạnh do các sự kiện đơn lẻ (cháy rừng, bụi); đây là giá trị thật nên được giữ, nhưng không phù hợp làm thước đo chính (xem từ điển dữ liệu, mục 13).
- `Disease_AdjPrev` ở bảng này được tính gộp COPD và Asthma (hai bệnh có mức nền khác nhau) nên gần như không có outlier; ở mục 9.3 ngưỡng cho bệnh được tính **riêng theo từng bệnh**.
- Ngưỡng dùng để đánh dấu `is_extreme_*` được tính lại ở mục 9.3 trên bộ dữ liệu cuối (sau khi lọc độ phủ AQI và City), nên có thể lệch nhẹ so với bảng này.

6. KIỂM TRA ĐỘ PHỦ DỮ LIỆU AQI

In [51]:
# 6.1 Tính độ phủ dữ liệu AQI

df["Days_in_Year"] = df["Year"].apply(
    lambda year: 366 if year % 4 == 0 else 365
)

df["AQI_Coverage_Pct"] = (
    df["Days_with_AQI"] / df["Days_in_Year"] * 100
)

print(df["AQI_Coverage_Pct"].describe())

count    7566.000000
mean       89.062666
std        20.367261
min         1.917808
25%        92.054795
50%        99.180328
75%       100.000000
max       100.000000
Name: AQI_Coverage_Pct, dtype: float64


In [52]:
# 6.2 Kiểm tra mức độ phủ AQI

print("Coverage < 50%:", (df["AQI_Coverage_Pct"] < 50).sum())
print("Coverage < 75%:", (df["AQI_Coverage_Pct"] < 75).sum())
print("Coverage < 90%:", (df["AQI_Coverage_Pct"] < 90).sum())
print("Coverage >= 90%:", (df["AQI_Coverage_Pct"] >= 90).sum())

Coverage < 50%: 570
Coverage < 75%: 1442
Coverage < 90%: 1796
Coverage >= 90%: 5770


In [53]:
# 6.3 Lọc dữ liệu có độ phủ AQI >= MIN_COVERAGE_PCT

rows_before = len(df)

df = df[df["AQI_Coverage_Pct"] >= MIN_COVERAGE_PCT].copy()

rows_after = len(df)

print("Ngưỡng độ phủ:", MIN_COVERAGE_PCT, "%")
print("Trước khi lọc:", rows_before)
print("Số dòng bị loại:", rows_before - rows_after)
print("Sau khi lọc:", rows_after)
print("Tỷ lệ loại:", round((rows_before - rows_after) / rows_before * 100, 2), "%")

pipeline_log.append((f"Lọc độ phủ AQI >= {MIN_COVERAGE_PCT}%", len(df), df["CountyFIPS"].nunique()))

Ngưỡng độ phủ: 50 %
Trước khi lọc: 7566
Số dòng bị loại: 570
Sau khi lọc: 6996
Tỷ lệ loại: 7.53 %


7. KIỂM TRA TÍNH ĐẦY ĐỦ THEO NĂM

In [54]:
# 7.1 Kiểm tra số năm dữ liệu của mỗi County

county_year_count = df.groupby("CountyFIPS")["Year"].nunique()

print("Tổng số County:", len(county_year_count))

print("\nSố County theo số năm dữ liệu:")
print(
    county_year_count
    .value_counts()
    .sort_index(ascending=False)
)

Tổng số County: 939

Số County theo số năm dữ liệu:
Year
4    740
3    159
2     21
1     19
Name: count, dtype: int64


In [55]:
# 7.2 Đánh dấu County có đủ dữ liệu 4 năm

county_year_count = df.groupby("CountyFIPS")["Year"].nunique()

complete_counties = county_year_count[
    county_year_count == 4
].index

df["Complete_4_Years"] = df["CountyFIPS"].isin(complete_counties)

print(df["Complete_4_Years"].value_counts())
print("\nShape:", df.shape)

Complete_4_Years
True     5920
False    1076
Name: count, dtype: int64

Shape: (6996, 27)


8. BỔ SUNG BIẾN CÁC THÀNH PHỐ LỚN Ở MỸ

In [56]:
# 8.1 Đọc dữ liệu AQS Sites

aqs_sites = pd.read_csv(AQS_SITES_FILE, low_memory=False)

print("Shape:", aqs_sites.shape)
print("\nColumns:")
print(aqs_sites.columns.tolist())

Shape: (20994, 28)

Columns:
['State Code', 'County Code', 'Site Number', 'Latitude', 'Longitude', 'Datum', 'Elevation', 'Land Use', 'Location Setting', 'Site Established Date', 'Site Closed Date', 'Met Site State Code', 'Met Site County Code', 'Met Site Site Number', 'Met Site Type', 'Met Site Distance', 'Met Site Direction', 'GMT Offset', 'Owning Agency', 'Local Site Name', 'Address', 'Zip Code', 'State Name', 'County Name', 'City Name', 'CBSA Name', 'Tribe Name', 'Extraction Date']


Giải thích biến CountyFIPS

CountyFIPS là mã định danh duy nhất cho mỗi county (quận) tại Hoa Kỳ theo hệ thống mã FIPS. Mã này được hình thành từ:

State Code: mã của bang.
County Code: mã của county trong bang đó.

Trong bộ dữ liệu chính đã có biến CountyFIPS, trong khi file aqs_sites.csv lưu riêng State Code và County Code. Vì vậy, cần tạo CountyFIPS cho dữ liệu AQS Sites để có một khóa chung (key) giữa hai bộ dữ liệu.

Mục đích chính là:

Dataset chính (CountyFIPS) -> AQS Sites (CountyFIPS) -> xác định City Name tương ứng

Nhờ đó có thể bổ sung thông tin City vào bộ dữ liệu chính mà không phải ghép theo tên County hoặc State, giúp việc liên kết dữ liệu chính xác và nhất quán hơn.

In [57]:
# 8.2 Tạo CountyFIPS và kiểm tra mức độ khớp

aqs_sites["CountyFIPS"] = (
    pd.to_numeric(aqs_sites["State Code"], errors="coerce") * 1000
    + pd.to_numeric(aqs_sites["County Code"], errors="coerce")
)

aqs_sites["CountyFIPS"] = pd.to_numeric(
    aqs_sites["CountyFIPS"], errors="coerce"
).astype("Int64")

df["CountyFIPS"] = pd.to_numeric(
    df["CountyFIPS"], errors="coerce"
).astype("Int64")

data_counties = set(df["CountyFIPS"].dropna())
aqs_counties = set(aqs_sites["CountyFIPS"].dropna())

matched_counties = data_counties.intersection(aqs_counties)

print("County trong dataset chính:", len(data_counties))
print("County trong AQS Sites:", len(aqs_counties))
print("County khớp:", len(matched_counties))
print("Tỷ lệ khớp:",
      round(len(matched_counties) / len(data_counties) * 100, 2), "%")

print("\nSố dòng dataset chính:", len(df))
print("Số dòng có CountyFIPS khớp AQS:",
      df["CountyFIPS"].isin(aqs_counties).sum())
print("Số dòng không khớp:",
      (~df["CountyFIPS"].isin(aqs_counties)).sum())

County trong dataset chính: 939
County trong AQS Sites: 2133
County khớp: 939
Tỷ lệ khớp: 100.0 %

Số dòng dataset chính: 6996
Số dòng có CountyFIPS khớp AQS: 6996
Số dòng không khớp: 0


Làm sạch thông tin City: Sau khi liên kết bằng CountyFIPS, dữ liệu AQS Sites được giới hạn còn các county xuất hiện trong bộ dữ liệu chính. Biến City Name được chuẩn hóa bằng cách loại khoảng trắng thừa và xác định các bản ghi có tên thành phố hợp lệ. Các giá trị như 'Not in a city', 'Not in city' hoặc giá trị thiếu không được xem là thông tin thành phố hợp lệ.

In [58]:
# 8.3 Làm sạch thông tin City

aqs_relevant = aqs_sites[
    aqs_sites["CountyFIPS"].isin(data_counties)
].copy()

aqs_relevant["City_Clean"] = (
    aqs_relevant["City Name"]
    .astype("string")
    .str.strip()
)

aqs_relevant["Has_City"] = (
    aqs_relevant["City_Clean"].notna()
    & ~aqs_relevant["City_Clean"].str.lower().isin([
        "not in a city",
        "not in city"
    ])
)

city_check = (
    aqs_relevant
    .groupby("CountyFIPS")["Has_City"]
    .any()
)

print("County có thông tin City:", city_check.sum())
print("County không có thông tin City:", (~city_check).sum())
print("Tổng:", len(city_check))

County có thông tin City: 858
County không có thông tin City: 81
Tổng: 939


In [59]:
# 8.4 Kiểm tra số City trong mỗi County

aqs_city = aqs_relevant[
    aqs_relevant["Has_City"] == True
].copy()

county_city_check = (
    aqs_city
    .groupby("CountyFIPS")["City_Clean"]
    .nunique()
)

print("County có 1 City:",
      (county_city_check == 1).sum())

print("County có nhiều hơn 1 City:",
      (county_city_check > 1).sum())

print("Số City khác nhau trong AQS:",
      aqs_city["City_Clean"].nunique())

County có 1 City: 292
County có nhiều hơn 1 City: 566
Số City khác nhau trong AQS: 2857


Lựa chọn City đại diện: Do một County có thể chứa nhiều City, mỗi County được gán một City đại diện dựa trên số lượng trạm quan trắc AQS. City có số trạm AQS nhiều nhất trong County được chọn làm City đại diện. Cách này giúp đảm bảo mỗi CountyFIPS chỉ tương ứng với một City trước khi merge, tránh làm tăng số dòng của bộ dữ liệu chính.

In [60]:
# 8.5 Chọn City đại diện cho mỗi County

city_counts = (
    aqs_city
    .groupby(["CountyFIPS", "City_Clean"])
    .size()
    .reset_index(name="Num_Sites")
)

city_counts = city_counts.sort_values(
    ["CountyFIPS", "Num_Sites"],
    ascending=[True, False]
)

county_city = (
    city_counts
    .drop_duplicates(subset="CountyFIPS")
    [["CountyFIPS", "City_Clean", "Num_Sites"]]
    .rename(columns={
        "City_Clean": "City",
        "Num_Sites": "City_AQS_Sites"
    })
)

print("Số County có City đại diện:", len(county_city))
print("Số City đại diện khác nhau:", county_city["City"].nunique())

county_city.head(10)

Số County có City đại diện: 858
Số City đại diện khác nhau: 792


,CountyFIPS,City,City_AQS_Sites
0,1003,Fairhope,1
2,1033,Muscle Shoals,3
5,1049,Fort Payne,6
9,1051,Wetumpka,4
11,1055,Gadsden,7
13,1069,Dothan,4
15,1073,Birmingham,41
27,1089,Huntsville,19
32,1097,Mobile,30
36,1101,Montgomery,8


Bổ sung City vào dữ liệu chính: Sau khi xác định một City đại diện cho mỗi County, dữ liệu City được liên kết với bộ dữ liệu chính thông qua CountyFIPS. Sử dụng left join để giữ nguyên toàn bộ dữ liệu hiện có và kiểm tra các County chưa xác định được City đại diện.

In [61]:
# 8.6 Merge City vào dataset chính

rows_before = len(df)

df_city = df.merge(
    county_city[["CountyFIPS", "City"]],
    on="CountyFIPS",
    how="left"
)

print("Số dòng trước merge:", rows_before)
print("Số dòng sau merge:", len(df_city))
print("Số dòng có City:", df_city["City"].notna().sum())
print("Số dòng không có City:", df_city["City"].isna().sum())

Số dòng trước merge: 6996
Số dòng sau merge: 6996
Số dòng có City: 6406
Số dòng không có City: 590


Xử lý các quan sát không có City: Sau khi merge, có một số quan sát không xác định được City đại diện từ AQS Sites. Vì biến City cần thiết cho mục tiêu trực quan hóa dữ liệu theo thành phố, các quan sát này được loại khỏi bộ dữ liệu cuối. Số dòng và số County bị loại ở bước này được in ở output bên dưới và ghi trong bảng theo dõi số dòng (mục 11.3), nên số liệu luôn khớp với lần chạy hiện tại.

In [62]:
# 8.7 Giữ các quan sát có City đại diện

rows_before, counties_before = len(df_city), df_city["CountyFIPS"].nunique()

df = df_city[df_city["City"].notna()].copy()

print("Số dòng bị loại vì không có City:", rows_before - len(df))
print("Số County bị loại vì không có City:", counties_before - df["CountyFIPS"].nunique())

print("\nShape:", df.shape)
print("Số County:", df["CountyFIPS"].nunique())
print("Số City:", df["City"].nunique())

print("\nSố dòng theo Disease:")
print(df["Disease"].value_counts())

print("\nSố dòng theo Year:")
print(df["Year"].value_counts().sort_index())

pipeline_log.append(("Giữ County có City đại diện", len(df), df["CountyFIPS"].nunique()))

Số dòng bị loại vì không có City: 590
Số County bị loại vì không có City: 81

Shape: (6406, 28)
Số County: 858
Số City: 792

Số dòng theo Disease:
Disease
COPD              3203
Current Asthma    3203
Name: count, dtype: int64

Số dòng theo Year:
Year
2019    1622
2020    1654
2021    1604
2022    1526
Name: count, dtype: int64


9. TẠO BIẾN MỚI VÀ ĐÁNH DẤU OUTLIER

Tạo biến Poor Air Days: Poor_Air_Days thể hiện tổng số ngày trong năm mà AQI thuộc các mức Unhealthy for Sensitive Groups (USG), Unhealthy, Very Unhealthy hoặc Hazardous. Biến Poor_Air_Days_Pct biểu diễn tỷ lệ các ngày chất lượng không khí kém trên tổng số ngày có dữ liệu AQI, giúp so sánh giữa các khu vực có số ngày quan trắc khác nhau.

In [63]:
# 9.1 Tạo biến Poor Air Days

df["Poor_Air_Days"] = (
    df["USG_Days"]
    + df["Unhealthy_Days"]
    + df["Very_Unhealthy_Days"]
    + df["Hazardous_Days"]
)

df["Poor_Air_Days_Pct"] = (
    df["Poor_Air_Days"] / df["Days_with_AQI"] * 100
)

print("Shape:", df.shape)
print("Missing Poor_Air_Days:", df["Poor_Air_Days"].isna().sum())
print("Missing Poor_Air_Days_Pct:", df["Poor_Air_Days_Pct"].isna().sum())

df[[
    "City",
    "Year",
    "Days_with_AQI",
    "Poor_Air_Days",
    "Poor_Air_Days_Pct"
]].head()

Shape: (6406, 30)
Missing Poor_Air_Days: 0
Missing Poor_Air_Days_Pct: 0


,City,Year,Days_with_AQI,Poor_Air_Days,Poor_Air_Days_Pct
0,Fairhope,2019,271,0,0.0
1,Fairhope,2019,271,0,0.0
2,Muscle Shoals,2019,263,0,0.0
3,Muscle Shoals,2019,263,0,0.0
4,Fort Payne,2019,361,0,0.0


Tách tọa độ địa lý: Biến Geolocation lưu tọa độ dưới dạng POINT (Longitude Latitude). Longitude và Latitude được tách thành hai biến số riêng để thuận tiện cho việc trực quan hóa dữ liệu trên bản đồ. Các tọa độ này đại diện cho vị trí địa lý của County trong dữ liệu gốc, không phải tọa độ trung tâm của City đại diện.

Longitude/Latitude trong PLACES thay đổi nhẹ giữa các năm (bản phát hành khác nhau); để mỗi County có **một** vị trí cố định khi vẽ bản đồ, lấy trung vị theo County.

In [64]:
# 9.2 Tách Longitude và Latitude

coords = df["Geolocation"].str.extract(
    r"POINT \(([-\d.]+) ([-\d.]+)\)"
)

df["Longitude"] = pd.to_numeric(coords[0], errors="coerce")
df["Latitude"]  = pd.to_numeric(coords[1], errors="coerce")

print("Missing Longitude:", df["Longitude"].isna().sum())
print("Missing Latitude:", df["Latitude"].isna().sum())

# Toạ độ PLACES đổi nhẹ theo năm -> cố định 1 vị trí cho mỗi County (trung vị các năm)
n_changed = (df.groupby("CountyFIPS")["Latitude"].nunique() > 1).sum()
df[["Latitude", "Longitude"]] = df.groupby("CountyFIPS")[["Latitude", "Longitude"]].transform("median")
print("County có toạ độ đổi theo năm (trước khi cố định):", n_changed)
print("County có toạ độ đổi theo năm (sau khi cố định):",
      (df.groupby("CountyFIPS")["Latitude"].nunique() > 1).sum())
print("Shape:", df.shape)

df[["City", "County", "State", "Latitude", "Longitude"]].head()

Missing Longitude: 0
Missing Latitude: 0
County có toạ độ đổi theo năm (trước khi cố định): 840
County có toạ độ đổi theo năm (sau khi cố định): 0
Shape: (6406, 32)


,City,County,State,Latitude,Longitude
0,Fairhope,Baldwin,Alabama,30.693482,-87.734410
1,Fairhope,Baldwin,Alabama,30.693482,-87.734410
2,Muscle Shoals,Colbert,Alabama,34.700849,-87.804803
3,Muscle Shoals,Colbert,Alabama,34.700849,-87.804803
4,Fort Payne,DeKalb,Alabama,34.460559,-85.803995


**Đánh dấu outlier (không xóa dòng):** Các giá trị vượt ngưỡng IQR (Q1 − 1.5·IQR, Q3 + 1.5·IQR) được **đánh dấu** bằng các cột boolean `is_extreme_aqi`, `is_extreme_smoking`, `is_extreme_disease`, `is_extreme_population`, không dùng để lọc dòng. Các giá trị cực trị phản ánh sự khác biệt thực tế giữa các khu vực và là phần quan trọng của phân tích; xóa chúng sẽ làm mất đúng những quan sát đáng quan tâm nhất. Các cột đánh dấu cho phép bước EDA và mô hình sau này kiểm tra độ nhạy (so sánh kết quả có và không có các dòng extreme, Cook's Distance).

Cờ AQI hai phía: `is_extreme_aqi` = AQI **cao bất thường** (`is_extreme_aqi_high`, thường là các county ô nhiễm nặng) **hoặc thấp bất thường** (`is_extreme_aqi_low`, các county rất sạch hoặc có AQI bị chi phối bởi một chất ô nhiễm ít gặp). Hai phía này có ý nghĩa khác nhau nên được tách riêng.

Ngưỡng được tính trên **bộ dữ liệu phân tích cuối cùng** (sau khi lọc độ phủ AQI và City) để "extreme" nghĩa là cực đoan so với chính mẫu sẽ được phân tích. Ngưỡng của `Disease_AdjPrev` được tính **riêng cho từng bệnh** vì COPD và Current Asthma có thang đo khác nhau. Riêng `Population` được giữ nguyên và bổ sung `Log_Population` để giảm độ lệch phải; không xóa các county lớn vì đây là những đơn vị quan trọng nhất của bài toán.

In [65]:
# 9.3 Đánh dấu outlier trên bộ dữ liệu phân tích cuối (KHÔNG xóa dòng)
# (dùng lại hàm iqr_bounds đã định nghĩa ở mục 5.1)

def outside(s, lo, hi):
    return (s < lo) | (s > hi)

bounds = {c: iqr_bounds(df[c]) for c in
          ["Median_AQI", "AQI_90th_Percentile", "Smoking_AdjPrev", "Population"]}

# AQI: tách phía cao và phía thấp
df["is_extreme_aqi_high"] = ((df["Median_AQI"] > bounds["Median_AQI"][1])
                             | (df["AQI_90th_Percentile"] > bounds["AQI_90th_Percentile"][1]))
df["is_extreme_aqi_low"]  = ((df["Median_AQI"] < bounds["Median_AQI"][0])
                             | (df["AQI_90th_Percentile"] < bounds["AQI_90th_Percentile"][0]))
df["is_extreme_aqi"]      = df["is_extreme_aqi_high"] | df["is_extreme_aqi_low"]

df["is_extreme_smoking"]    = outside(df["Smoking_AdjPrev"], *bounds["Smoking_AdjPrev"])
df["is_extreme_population"] = outside(df["Population"], *bounds["Population"])
df["Log_Population"]        = np.log10(df["Population"])

# Disease: COPD và Asthma khác thang đo -> tính ngưỡng riêng theo từng bệnh
disease_bounds = {d: iqr_bounds(g["Disease_AdjPrev"]) for d, g in df.groupby("Disease")}
df["is_extreme_disease"] = (
    df.groupby("Disease")["Disease_AdjPrev"]
      .transform(lambda s: outside(s, *iqr_bounds(s)))
)

flag_cols = ["is_extreme_aqi", "is_extreme_aqi_high", "is_extreme_aqi_low",
             "is_extreme_smoking", "is_extreme_disease", "is_extreme_population"]

print("Ngưỡng IQR trên dữ liệu cuối (Lower, Upper):")
for c, (lo, hi) in bounds.items():
    print(f"  {c}: ({lo:.2f}, {hi:.2f})")
for d, (lo, hi) in disease_bounds.items():
    print(f"  Disease_AdjPrev [{d}]: ({lo:.2f}, {hi:.2f})")

print("\nSố dòng (không đổi):", len(df))
print("\nSố dòng bị đánh dấu:")
print(df[flag_cols].sum())
print("\nTỷ lệ (%):")
print((df[flag_cols].mean() * 100).round(2))
print("\nis_extreme_disease theo bệnh:")
print(df.groupby("Disease")["is_extreme_disease"].sum())

Ngưỡng IQR trên dữ liệu cuối (Lower, Upper):
  Median_AQI: (21.00, 61.00)
  AQI_90th_Percentile: (32.50, 84.50)
  Smoking_AdjPrev: (5.80, 27.40)
  Population: (-364152.38, 737142.62)
  Disease_AdjPrev [COPD]: (2.55, 10.15)
  Disease_AdjPrev [Current Asthma]: (7.55, 12.75)

Số dòng (không đổi): 6406

Số dòng bị đánh dấu:
is_extreme_aqi           540
is_extreme_aqi_high      314
is_extreme_aqi_low       226
is_extreme_smoking        38
is_extreme_disease        78
is_extreme_population    662
dtype: int64

Tỷ lệ (%):
is_extreme_aqi            8.43
is_extreme_aqi_high       4.90
is_extreme_aqi_low        3.53
is_extreme_smoking        0.59
is_extreme_disease        1.22
is_extreme_population    10.33
dtype: float64

is_extreme_disease theo bệnh:
Disease
COPD              45
Current Asthma    33
Name: is_extreme_disease, dtype: int64


10. CHỌN VÀ SẮP XẾP CÁC BIẾN CUỐI CÙNG 

Lựa chọn biến cuối cùng: Sau quá trình tiền xử lý, các biến cần thiết cho phân tích và trực quan hóa được giữ lại và sắp xếp theo từng nhóm thông tin. Các biến trung gian hoặc đã được chuyển đổi như Geolocation và Days_in_Year không được đưa vào bộ dữ liệu cuối.

In [66]:
# 10.1 Chọn các biến cho dataset cuối

columns_final = [
    "CountyFIPS",
    "City",
    "County",
    "StateAbbr",
    "State",
    "Latitude",
    "Longitude",
    "Year",
    "Population",
    "Log_Population",

    "Days_with_AQI",
    "Good_Days",
    "Moderate_Days",
    "USG_Days",
    "Unhealthy_Days",
    "Very_Unhealthy_Days",
    "Hazardous_Days",

    "Max_AQI",
    "AQI_90th_Percentile",
    "Median_AQI",

    "Days_NO2",
    "Days_Ozone",
    "Days_PM2_5",
    "Days_PM10",

    "AQI_Coverage_Pct",
    "Poor_Air_Days",
    "Poor_Air_Days_Pct",

    "Smoking_AdjPrev",
    "Disease",
    "Disease_AdjPrev",

    "Complete_4_Years",

    "is_extreme_aqi",
    "is_extreme_aqi_high",
    "is_extreme_aqi_low",
    "is_extreme_smoking",
    "is_extreme_disease",
    "is_extreme_population"
]

df_final = df[columns_final].copy()

print("Shape final:", df_final.shape)
print("\nColumns:")
print(df_final.columns.tolist())

Shape final: (6406, 37)

Columns:
['CountyFIPS', 'City', 'County', 'StateAbbr', 'State', 'Latitude', 'Longitude', 'Year', 'Population', 'Log_Population', 'Days_with_AQI', 'Good_Days', 'Moderate_Days', 'USG_Days', 'Unhealthy_Days', 'Very_Unhealthy_Days', 'Hazardous_Days', 'Max_AQI', 'AQI_90th_Percentile', 'Median_AQI', 'Days_NO2', 'Days_Ozone', 'Days_PM2_5', 'Days_PM10', 'AQI_Coverage_Pct', 'Poor_Air_Days', 'Poor_Air_Days_Pct', 'Smoking_AdjPrev', 'Disease', 'Disease_AdjPrev', 'Complete_4_Years', 'is_extreme_aqi', 'is_extreme_aqi_high', 'is_extreme_aqi_low', 'is_extreme_smoking', 'is_extreme_disease', 'is_extreme_population']


11. KIỂM TRA LẠI DATASET CUỐI CÙNG

Kiểm tra dữ liệu cuối: Trước khi xuất dữ liệu, thực hiện kiểm tra lại giá trị thiếu, dòng trùng lặp, khóa County-Year-Disease, giá trị âm, tính nhất quán của số ngày AQI và phạm vi của các biến tỷ lệ.

In [67]:
# 11.1 Kiểm tra chất lượng dataset cuối

print("Shape:", df_final.shape)

print("\nTổng missing:",
      df_final.isnull().sum().sum())

print("Dòng trùng hoàn toàn:",
      df_final.duplicated().sum())

print("Trùng County-Year-Disease:",
      df_final.duplicated(
          subset=["CountyFIPS", "Year", "Disease"]
      ).sum())

numeric_check = [
    "Population",
    "Days_with_AQI",
    "Good_Days",
    "Moderate_Days",
    "USG_Days",
    "Unhealthy_Days",
    "Very_Unhealthy_Days",
    "Hazardous_Days",
    "Max_AQI",
    "AQI_90th_Percentile",
    "Median_AQI",
    "Days_NO2",
    "Days_Ozone",
    "Days_PM2_5",
    "Days_PM10",
    "Poor_Air_Days",
    "Poor_Air_Days_Pct"
]

print("\nSố giá trị âm:")
print((df_final[numeric_check] < 0).sum())

aqi_day_sum = (
    df_final["Good_Days"]
    + df_final["Moderate_Days"]
    + df_final["USG_Days"]
    + df_final["Unhealthy_Days"]
    + df_final["Very_Unhealthy_Days"]
    + df_final["Hazardous_Days"]
)

print("\nAQI category không khớp Days_with_AQI:",
      (aqi_day_sum != df_final["Days_with_AQI"]).sum())

print("Smoking ngoài 0-100:",
      ((df_final["Smoking_AdjPrev"] < 0) |
       (df_final["Smoking_AdjPrev"] > 100)).sum())

print("Disease ngoài 0-100:",
      ((df_final["Disease_AdjPrev"] < 0) |
       (df_final["Disease_AdjPrev"] > 100)).sum())

print(f"AQI Coverage ngoài {MIN_COVERAGE_PCT}-100:",
      ((df_final["AQI_Coverage_Pct"] < MIN_COVERAGE_PCT) |
       (df_final["AQI_Coverage_Pct"] > 100)).sum())

# Kiểm tra các cột đánh dấu outlier
print("\nKiểu dữ liệu cột đánh dấu:")
print(df_final[["is_extreme_aqi", "is_extreme_smoking",
                "is_extreme_disease", "is_extreme_population"]].dtypes)
print("Log_Population hữu hạn:", np.isfinite(df_final["Log_Population"]).all())
print("\nSố dòng extreme (dòng vẫn được giữ trong dữ liệu):")
print(df_final[["is_extreme_aqi", "is_extreme_smoking",
                "is_extreme_disease", "is_extreme_population"]].sum())

Shape: (6406, 37)

Tổng missing: 0
Dòng trùng hoàn toàn: 0
Trùng County-Year-Disease: 0

Số giá trị âm:
Population             0
Days_with_AQI          0
Good_Days              0
Moderate_Days          0
USG_Days               0
Unhealthy_Days         0
Very_Unhealthy_Days    0
Hazardous_Days         0
Max_AQI                0
AQI_90th_Percentile    0
Median_AQI             0
Days_NO2               0
Days_Ozone             0
Days_PM2_5             0
Days_PM10              0
Poor_Air_Days          0
Poor_Air_Days_Pct      0
dtype: int64

AQI category không khớp Days_with_AQI: 0
Smoking ngoài 0-100: 0
Disease ngoài 0-100: 0
AQI Coverage ngoài 50-100: 0

Kiểu dữ liệu cột đánh dấu:
is_extreme_aqi           bool
is_extreme_smoking       bool
is_extreme_disease       bool
is_extreme_population    bool
dtype: object
Log_Population hữu hạn: True

Số dòng extreme (dòng vẫn được giữ trong dữ liệu):
is_extreme_aqi           540
is_extreme_smoking        38
is_extreme_disease        78
is_extreme_

11.2 Bảng theo dõi số dòng qua từng bước

In [ ]:
# 11.2 Bảng theo dõi số dòng
pipeline_log.append(("Dataset cuối (đã chọn cột)", len(df_final), df_final["CountyFIPS"].nunique()))
print(pd.DataFrame(pipeline_log, columns=["Bước", "Số dòng", "Số County"]).to_string(index=False))

                       Bước  Số dòng  Số County
      Đọc file Merged_Clean     7566       1009
      Lọc độ phủ AQI >= 50%     6996        939
Giữ County có City đại diện     6406        858
 Dataset cuối (đã chọn cột)     6406        858


12. XUẤT DỮ LIỆU CUỐI CÙNG

In [70]:
# 12. Xuất dataset cuối

output_file = OUTPUT_FILE

df_final.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu file:", output_file)
print("Shape:", df_final.shape)

Đã lưu file: AQI_Respiratory_Disease_Final.csv
Shape: (6406, 37)


13. TỪ ĐIỂN DỮ LIỆU CỦA FILE BÀN GIAO

**Đơn vị quan sát:** một dòng = một **County – Năm – Bệnh** (mỗi County-Năm có 2 dòng: `COPD` và `Current Asthma`). Các biến AQI, hút thuốc, dân số và toạ độ lặp lại y hệt trên 2 dòng này; khi vẽ/phân tích chỉ liên quan tới các biến đó, nên bỏ trùng theo (`CountyFIPS`, `Year`) để không đếm đôi.

| Nhóm | Cột | Ý nghĩa |
|---|---|---|
| Định danh | `CountyFIPS` | Mã FIPS 5 chữ số của County (State Code × 1000 + County Code) |
| | `City` | City đại diện của County (City có nhiều trạm AQS nhất); **không** phải toạ độ hay ranh giới của City và **không duy nhất** (một số tên City trùng giữa các County/bang, khi gắn nhãn dùng "City, StateAbbr") |
| | `County`, `State`, `StateAbbr` | Tên County, tên bang, mã bang |
| Địa lý | `Latitude`, `Longitude` | Toạ độ của **County** (PLACES), đã cố định một vị trí cho mỗi County (trung vị các năm); không phải toạ độ City đại diện |
| Thời gian | `Year` | Năm 2019–2022 |
| Dân số | `Population` | Dân số County (PLACES); thay đổi nhẹ theo năm, khi phân loại "đô thị lớn" nên dùng giá trị lớn nhất/trung bình theo County |
| | `Log_Population` | log10(Population), dùng khi vẽ/mô hình vì dân số lệch phải mạnh |
| AQI theo năm | `Days_with_AQI` | Số ngày có dữ liệu AQI trong năm |
| | `Good_Days`, `Moderate_Days`, `USG_Days`, `Unhealthy_Days`, `Very_Unhealthy_Days`, `Hazardous_Days` | Số ngày theo mức AQI (USG = Unhealthy for Sensitive Groups) |
| | `Max_AQI`, `AQI_90th_Percentile`, `Median_AQI` | AQI lớn nhất, phân vị 90 và trung vị trong năm. `Max_AQI` lệch phải rất mạnh (một số County-Năm > 500 do cháy rừng/bụi) nên không dùng làm thước đo chính; ưu tiên `Median_AQI`, `AQI_90th_Percentile`, `Poor_Air_Days_Pct` |
| | `Days_Ozone`, `Days_PM2_5`, `Days_NO2`, `Days_PM10` | Số ngày mà chất đó là chất ô nhiễm chính. Bốn cột này **không luôn cộng đủ** `Days_with_AQI` vì các chất khác (ví dụ CO) không được giữ lại |
| | `AQI_Coverage_Pct` | Độ phủ dữ liệu AQI = `Days_with_AQI` / số ngày trong năm × 100 |
| | `Poor_Air_Days` | Số ngày AQI > 100 (USG + Unhealthy + Very Unhealthy + Hazardous) |
| | `Poor_Air_Days_Pct` | `Poor_Air_Days` / `Days_with_AQI` × 100 |
| Sức khỏe (PLACES) | `Smoking_AdjPrev` | Tỷ lệ hút thuốc hiện tại (%), age-adjusted prevalence |
| | `Disease` | `COPD` hoặc `Current Asthma` |
| | `Disease_AdjPrev` | Tỷ lệ bệnh (%), age-adjusted prevalence; là ước lượng vùng nhỏ theo mô hình |
| Chất lượng dữ liệu | `Complete_4_Years` | `True` nếu County có đủ 4 năm (2019–2022) sau khi lọc |
| | `is_extreme_aqi_high` | `True` nếu `Median_AQI` hoặc `AQI_90th_Percentile` vượt ngưỡng IQR **trên** (AQI cao bất thường) |
| | `is_extreme_aqi_low` | `True` nếu `Median_AQI` hoặc `AQI_90th_Percentile` thấp hơn ngưỡng IQR **dưới** (AQI thấp bất thường) |
| | `is_extreme_aqi` | `is_extreme_aqi_high` **hoặc** `is_extreme_aqi_low` |
| | `is_extreme_smoking`, `is_extreme_population` | `True` nếu vượt ngưỡng IQR của biến tương ứng |
| | `is_extreme_disease` | `True` nếu `Disease_AdjPrev` vượt ngưỡng IQR **của riêng bệnh đó** |

**Lưu ý khi sử dụng:**
- Không có dòng nào bị xóa vì outlier; dùng các cột `is_extreme_*` để lọc/tô màu/kiểm tra độ nhạy khi cần.
- Chỉ County có City đại diện và độ phủ AQI ≥ `MIN_COVERAGE_PCT` được giữ lại; số dòng/County qua từng bước xem bảng ở mục 11.3.
- Tỷ lệ bệnh PLACES thay đổi theo năm nhưng biến thiên giữa các County lớn hơn nhiều biến thiên theo năm trong cùng County (mục 4.5–4.6); phân tích chính nên là so sánh giữa các County.